# II. Calcul des délais d'accès

Ce notebook part des données préparées par le notebook 1 et :
1. trouve la caserne la plus proche de chaque forêt ;
2. convertit la distance en délai ;
3. rattache chaque forêt à sa commune ;
4. calcule les indicateurs par commune et la typologie ;
5. enregistre les résultats pour le notebook 3.

**Hypothèse importante :** délai (min) = distance à vol d'oiseau (km) × 1,3 / 50 km/h × 60. 
Il s'agit d'un **délai théorique minimal de trajet**, naturellement qui présente des limites.

## 0. Les préliminaires

In [1]:
# Pour installer les librairies requises
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import sys

# je cherche la racine du projet (le dossier qui contient "modules")
racine_projet = None
for essai in [".", "..", "../.."]:
    if os.path.exists(os.path.join(essai, "modules")) and racine_projet is None:
        racine_projet = os.path.abspath(essai)
print("Racine du projet :", racine_projet)

# j'ajoute la racine aux endroits où Python cherche les modules
if racine_projet not in sys.path:
    sys.path.append(racine_projet)

# si je modifie un module, il est rechargé automatiquement
%load_ext autoreload
%autoreload 2

Racine du projet : c:\Users\User\Documents\Python\Groupe_KC_AHA_Python


In [2]:
import pandas as pd
import geopandas as gpd

from modules.intern.ats import parametres as p
from modules.intern.ats import calculs

## 1. Lecture des données préparées

In [3]:
fichier_prep = "sorties/donnees_preparees.gpkg"
communes = gpd.read_file(fichier_prep, layer="communes")
casernes = gpd.read_file(fichier_prep, layer="casernes")
points_forets = gpd.read_file(fichier_prep, layer="points_forets")
nb_incendies = pd.read_csv("sorties/nb_incendies_par_commune.csv", sep=";", dtype={"Code INSEE": str})

print("Communes :", len(communes), "| Casernes :", len(casernes), "| Forêts :", len(points_forets))

Communes : 944 | Casernes : 460 | Forêts : 117445


## 2. Caserne la plus proche et délai

In [4]:
resultat = calculs.caserne_plus_proche(points_forets, casernes)
resultat = calculs.distance_vers_delai(resultat, p.COEF_DETOUR, p.VITESSE_KMH)
resultat[["id_foret", "nom_caserne", "distance_km", "delai_min"]].head()

,id_foret,nom_caserne,distance_km,delai_min
0,0,Centre d'incendie et de secours de Robion,5.203166,8.116938
1,1,Centre d'incendie et de secours de Collobrières,1.340150,2.090634
2,2,Centre d'incendie et de secours de Cavalaire-s...,3.533191,5.511778
3,3,Centre d'incendie et de secours de Savines-le-Lac,4.374913,6.824864
4,4,Centre d'incendie et de secours d'Argentière-l...,5.638313,8.795768


## 3. Rattachement aux communes et indicateurs

In [5]:
forets_communes = calculs.rattacher_aux_communes(resultat, communes, p.COL_CODE, p.COL_NOM, p.COL_DEP)
forets_communes = calculs.classer_delais(forets_communes, p.BORNES_DELAI, p.NOMS_CLASSES_DELAI)
print("Forêts situées en PACA :", len(forets_communes))

carte_communes = calculs.statistiques_par_commune(forets_communes, communes, nb_incendies, p.COL_CODE)
carte_communes, seuil_incendies, seuil_delai = calculs.typologie_communes(carte_communes)
print("Seuils (médianes) : incendies =", seuil_incendies, "| délai =", round(seuil_delai, 1), "min")
print(carte_communes["typologie"].value_counts())

Forêts situées en PACA : 109498
Seuils (médianes) : incendies = 3.0 | délai = 6.4 min
typologie
Éloignée, peu de feux     317
Exposée mais desservie    260
Faible enjeu              211
Prioritaire               154
Pas de forêt                2
Name: count, dtype: int64


In [6]:
carte_communes[[p.COL_NOM, "nb_forets", "delai_moyen", "nb_incendies", "typologie"]].head()

,nom,nb_forets,delai_moyen,nb_incendies,typologie
0,Abriès-Ristolas,504.0,7.321417,0.0,"Éloignée, peu de feux"
1,Marseille,427.0,3.374107,120.0,Exposée mais desservie
2,Cannes,23.0,2.447641,5.0,Exposée mais desservie
3,Bormes-les-Mimosas,340.0,6.824948,20.0,Prioritaire
4,Saint-Paul-sur-Ubaye,299.0,11.775239,4.0,Prioritaire


## 4. Enregistrement des résultats

In [ ]:
fichier_res = "sorties/resultats.gpkg"
forets_communes.to_file(fichier_res, layer="forets_communes", driver="GPKG")
carte_communes.to_file(fichier_res, layer="carte_communes", driver="GPKG")
print("Les résultats sont enregistrés !")

Résultats enregistrés !


________________________